In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
data = pd.read_excel('Data_Train.xlsx')

In [ ]:
data.head()

In [ ]:
with pd.option_context('display.max_rows', None):
    print(data['Additional_Info'])

In [ ]:
data.shape

In [ ]:
data.isna().sum()

In [ ]:
data.dropna(inplace=True)

In [ ]:
data.isna().sum()

In [ ]:
data.shape

In [ ]:
data.info()

In [ ]:
#handling DOJ / DEP-TIME / ARRIVAL-TIME

def change_to_date(column):
    data[column] = pd.to_datetime(data[column],infer_datetime_format=True)

In [ ]:
data.columns

In [ ]:
for i in ['Date_of_Journey', 'Dep_Time', 'Arrival_Time']:
    change_to_date(i)

In [ ]:
data.head()

In [ ]:
data.dtypes

In [ ]:
data['Journey_date'] = data['Date_of_Journey'].dt.day
data['Journey_month'] = data['Date_of_Journey'].dt.month
data['Journey_year'] = data['Date_of_Journey'].dt.year

In [ ]:
data.head()

In [ ]:
data.info()

In [ ]:
data['Journey_year'].unique()

In [ ]:
#data['Arrival_hour'] = data['Arrival_time'].dt.hour
#data['Arrival_min'] = data['Arrival_time'].dt.min

#fun for this:
#getting hour and minute
def getting_hour(df,column):
    df[column+'_hour'] = df[column].dt.hour
def getting_minute(df,column):
    df[column+'_min'] = df[column].dt.minute

In [ ]:
#drop column
def drop_column(df,column):
    df.drop(column,axis=1, inplace = True)

In [ ]:
getting_hour(data,'Arrival_Time')
getting_minute(data,'Arrival_Time')
getting_hour(data,'Dep_Time')
getting_minute(data,'Dep_Time')

In [ ]:
data.head()

In [ ]:
data.info()

In [ ]:
drop_column(data,'Date_of_Journey')
drop_column(data,'Journey_year')
drop_column(data,'Dep_Time')
drop_column(data,'Arrival_Time')

In [ ]:
data.info()

In [ ]:
time_duration = list(data['Duration'])
#time_duration

In [ ]:
for i in range(len(time_duration)):
    if len(time_duration[i].split(' '))==2:
        pass
    else:
        if 'h' in time_duration[i]:
            time_duration[i] = time_duration[i] + ' 0m'
        else:
            time_duration[i] = '0h ' + time_duration[i]

In [ ]:
#time_duration

In [ ]:
data['Duration'] = time_duration

In [ ]:
data.head()

In [ ]:
def hour(i):
    return i.split(' ')[0][0:-1]

def minute(i):
    return i.split(' ')[1][0:-1]

In [ ]:
data['Duration_hour'] = data['Duration'].apply(hour).astype(int)
data['Duration_min']  = data['Duration'].apply(minute).astype(int)

In [ ]:
data.head()

In [ ]:
data.info()

In [ ]:
data['Duration_hour']

In [ ]:
data.dtypes

In [ ]:
#import os
#print(os.getcwd())

In [ ]:
drop_column(data,'Duration')

In [ ]:
data.head()

In [ ]:
data['Total_Stops'].unique()

In [ ]:
dict_a = {'non-stop' : 0 , '2 stops' : 2, '1 stop' : 1, '3 stops' : 3, '4 stops' : 4}

In [ ]:
data['Total_Stops'] = data['Total_Stops'].map(dict_a)

In [ ]:
data.head(2)

In [ ]:
data.info()

In [ ]:
data['Airline'].unique()

In [ ]:
len(data['Airline'].unique())

In [ ]:
object_cols = [columns for columns in data.columns if data[columns].dtype==object]

In [ ]:
object_cols

In [ ]:
object_cols = data[object_cols]

In [ ]:
object_cols

In [ ]:
sorted_cols = [columns for columns in data.columns if data[columns].dtype!=object]

In [ ]:
sorted_cols

In [ ]:
sorted_cols = data[sorted_cols]

In [ ]:
sorted_cols.head()

In [ ]:
import seaborn as sns

In [ ]:
#plotting
plt.figure(figsize=(20,5))
sns.boxplot(x='Airline' ,  y='Price', data=data.sort_values('Price', ascending=False))
plt.show()

In [ ]:
Airline = pd.get_dummies(object_cols['Airline'], drop_first=True)

In [ ]:
Airline.head()

In [ ]:
object_cols.columns

In [ ]:
object_cols['Source'].unique()

In [ ]:
Source = pd.get_dummies(object_cols['Source'], drop_first=True)

In [ ]:
object_cols['Destination'].unique()

In [ ]:
Destination = pd.get_dummies(object_cols['Destination'], drop_first=True)

In [ ]:
object_cols['Route'].head()

In [ ]:
#handling Route Column:

object_cols['Route1'] = object_cols['Route'].str.split(' → ').str[0]
object_cols['Route2'] = object_cols['Route'].str.split(' → ').str[1]
object_cols['Route3'] = object_cols['Route'].str.split(' → ').str[2]
object_cols['Route4'] = object_cols['Route'].str.split(' → ').str[3]
object_cols['Route5'] = object_cols['Route'].str.split(' → ').str[4]

In [ ]:
object_cols

In [ ]:
max_row = object_cols['Route'].str.len().idxmax()

In [ ]:
len_row = object_cols.loc[max_row]

In [ ]:
len_row

In [ ]:
object_cols['Route6'] = object_cols['Route'].str.split(' → ').str[5]

In [ ]:
object_cols.isnull().sum()

In [ ]:
drop_column(object_cols,'Route')
drop_column(object_cols, 'Route6')

In [ ]:
for i in ['Route3', 'Route4', 'Route5']:
    object_cols[i].fillna('None',inplace=True)

In [ ]:
object_cols.head(3)

In [ ]:
data['Additional_Info'].value_counts()

In [ ]:
drop_column(object_cols,'Additional_Info')

In [ ]:
object_cols

In [ ]:
from sklearn.preprocessing import LabelEncoder

In [ ]:
le = LabelEncoder()

In [ ]:
for i in ['Route1' , 'Route2' , 'Route3' , 'Route4' , 'Route5']:
    object_cols[i] = le.fit_transform(object_cols[i])

In [ ]:
object_cols['Route3'].unique()

In [ ]:
pd.options.display.max_columns=40

In [ ]:
#put Airline/Source/Destination to the dataframe
data2 = pd.concat([object_cols,Airline,Source,Destination,sorted_cols],axis=1)
data2

In [ ]:
drop_column(data2,'Airline')
drop_column(data2,'Source')
drop_column(data2,'Destination')

In [ ]:
data2.head()

In [ ]:
data2 = data2.astype(int)
data2.head()

In [ ]:
data2.dtypes

In [ ]:
data2.info()

In [ ]:
import matplotlib.pyplot as plt 
import seaborn as sns

In [ ]:
sns.boxplot(y=data2['Price'],x = data2['Route1'])

In [ ]:
data2['Price'] = np.where(data2['Price']>=30000,
                            data2['Price'].median(),
                            data2['Price'])

In [ ]:
sns.boxplot(y=data2['Price'],x = data2['Route1'])

In [ ]:
X = data2.drop('Price',axis=1)
y = data2['Price']

In [ ]:
X

In [ ]:
y

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.25,random_state=42)

In [ ]:
from sklearn.linear_model import LinearRegression

In [ ]:
lr = LinearRegression()

In [ ]:
lr.fit(X_train,y_train)

In [ ]:
y_pred = lr.predict(X_test)

In [ ]:
from sklearn.metrics import mean_squared_error,r2_score, mean_absolute_error

In [ ]:
mean_squared_error(y_test,y_pred)

In [ ]:
r2_score(y_test,y_pred)

In [ ]:
mean_absolute_error(y_test,y_pred)

In [ ]:
def algo(reg,X_train,y_train,X_test,y_test): 
    model = reg()
    model.fit(X_train,y_train)
    y_pred = model.predict(X_test)
    print(reg)
    print(mean_squared_error(y_test,y_pred))
    print(r2_score(y_test,y_pred))

In [ ]:
from sklearn.ensemble import RandomForestRegressor

In [ ]:
algo(RandomForestRegressor,X_train,y_train,X_test,y_test)